Poissonov problem $\\$
$-\Delta u = f$ na $(0,1)^2$ $\\$
$u=0$ na $\partial[0,1]^2$

In [ ]:
from ngsolve import *
from ngsolve.webgui import Draw

import scipy.sparse as sp
from scipy.sparse import csr_matrix
import matplotlib.pylab as plt

In [ ]:
mesh = Mesh(unit_square.GenerateMesh(maxh=0.1))
Draw (mesh);

In [ ]:
fes = H1(mesh, order=1, dirichlet="left|right|bottom|top")
print ("ndof =", fes.ndof)

In [ ]:
print (mesh.GetBoundaries())

In [ ]:
u = fes.TrialFunction()
v = fes.TestFunction()

f = LinearForm(fes)
f += 32*(y*(1-y)+x*(1-x))*v*dx

a = BilinearForm(fes)
a += InnerProduct(grad(u), grad(v)) * dx

a.Assemble()
f.Assemble()

In [ ]:
n = a.mat.height
density = a.mat.nze / (n*n)
print("Ukupan broj elemenata:", n*n)
print("Broj nenul elemenata:", a.mat.nze)
print("Popunjenost:", density)

In [ ]:
plt.rcParams['figure.figsize'] = (12, 12)
A = sp.csr_matrix(a.mat.CSR())

plt.spy(A)
plt.show()

In [ ]:
print (csr_matrix(a.mat.CSR()))

In [ ]:
solution_gf = GridFunction(fes)
solution_gf.vec.data = a.mat.Inverse(fes.FreeDofs()) * f.vec

flux_gf = -grad(solution_gf)

In [ ]:
Draw (solution_gf, mesh, deformation=True, scale=0.5)

In [ ]:
Draw (flux_gf, mesh, vectors= {"grid_size" : 40});

In [ ]:
exact_cf = 16*x*(1-x)*y*(1-y)

error_L2 = sqrt(Integrate((solution_gf - exact_cf)**2, mesh))
print ("L2 norm error:", error_L2)

In [ ]:
exact_grad_cf_x = exact_cf.Diff(x)
exact_grad_cf_y = exact_cf.Diff(y)

exact_grad_cf = CF((exact_grad_cf_x, exact_grad_cf_y))

error_H1_semi = sqrt(Integrate((grad(solution_gf) - exact_grad_cf)**2, mesh))
print ("H1 seminorm error:", error_H1_semi)

In [ ]:
error_H1 = sqrt(error_L2**2 + error_H1_semi**2)
print ("H1 norm error:", error_H1)